# 05 · Análise: respostas às perguntas de negócio

**Problema:** a Olist conecta pequenos vendedores a grandes marketplaces. Ela não controla o estoque
nem a transportadora, mas é avaliada pelo cliente final. O atraso na entrega destrói a satisfação?
E esse atraso é aleatório ou se concentra em padrões que podem ser gerenciados?

**Hipótese central:** *o atraso é o principal fator de insatisfação e se concentra em rotas,
períodos e elos da cadeia identificáveis. Logo, é gerenciável com dados.*

| # | Pergunta | Métrica principal |
|---|---|---|
| P1 | Quanto o atraso custa em satisfação? | Nota média e % de notas baixas (1–2), com e sem atraso |
| P2 | Onde o atraso se concentra (UF, região, rota, distância)? | Taxa de atraso por recorte |
| P3 | Quando o atraso acontece? Está ligado a picos de demanda? | Taxa de atraso mensal × volume |
| P4 | Quem causa o atraso: o vendedor (postagem) ou o transporte? | Participação de cada elo nos atrasos |
| P5 | O prazo prometido está calibrado? | Folga (prometido − real) e simulação de promessa alternativa |
| P6 | Dá para prever o risco de atraso no momento da compra? *(exploratória)* | AUC e *lift* fora da amostra |

**Base de análise (regra A01, ver notebook 04):** pedidos entregues de forma válida
(`flag_entregue_valido`) com compra entre **jan/2017 e ago/2018**. **Taxa de atraso** = % de
pedidos entregues depois da data prometida (regra G03).

### Configuração do ambiente
Esta célula prepara o ambiente de execução. No Google Colab, ela:
1. clona o repositório do projeto, que contém o módulo `scripts/common.py`;
2. instala o PySpark e o Delta Lake;
3. monta o Google Drive, onde o lakehouse fica persistido.

Na primeira execução, o Colab pede autorização de acesso ao Drive.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/FilippeDumont/mvp-olist-atrasos-entrega.git"
REPO_DIR = "/content/repo"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub"], check=True)
    sys.path.insert(0, f"{REPO_DIR}/scripts")
else:  # execução local de teste
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import common as C

C.mount_drive()
spark = C.get_spark()
print("Spark", spark.version, "| Delta Lake habilitado")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
import graficos as G

C.banner("05 · Análise das perguntas de negócio")
pd.set_option("display.width", 200, "display.max_columns", 30)
DESTINOS = [f"{C.CLOUD_BASE}/evidencias/graficos", os.path.join(C.REPO_ROOT, "evidencias", "graficos")]
C.register_views("gold", ["fato_pedido", "dim_tempo", "dim_cliente", "dim_vendedor", "dim_produto"])
sql = lambda q: spark.sql(q).toPandas()

spark.sql("""
CREATE OR REPLACE TEMP VIEW base AS
SELECT f.*,
       tc.data AS data_compra, tc.ano_mes, tc.nome_dia_semana, tc.dia_semana, tc.mes,
       te.data AS data_entrega,
       datediff(te.data, tc.data) AS dias_corridos_entrega,
       c.uf AS uf_cliente, c.regiao AS regiao_cliente,
       v.uf AS uf_vendedor, v.regiao AS regiao_vendedor,
       p.categoria, p.peso_g, p.volume_cm3
FROM gold_fato_pedido f
JOIN gold_dim_tempo   tc ON f.sk_data_compra  = tc.sk_data
JOIN gold_dim_tempo   te ON f.sk_data_entrega = te.sk_data
JOIN gold_dim_cliente c  ON f.sk_cliente = c.sk_cliente
JOIN gold_dim_vendedor v ON f.sk_vendedor = v.sk_vendedor
JOIN gold_dim_produto p  ON f.sk_produto = p.sk_produto
WHERE f.flag_entregue_valido
  AND f.sk_data_compra BETWEEN 20170101 AND 20180831     -- regra A01
""")
df = spark.table("base").toPandas()
df["atraso"] = df.flag_atraso.astype(int)
TAXA = 100 * df.atraso.mean()
print(f"Base de análise: {len(df):,} pedidos | taxa de atraso geral: {TAXA:.2f}%")
INDICADORES = []   # resumo persistido ao final

---
## P1 · Quanto o atraso custa em satisfação?
*Hipótese:* pedidos atrasados recebem notas significativamente piores.

In [ ]:
p1 = sql("""
SELECT CASE WHEN flag_atraso THEN 'Atrasado' ELSE 'No prazo' END AS entrega,
       count(*) AS pedidos_avaliados,
       round(avg(nota_avaliacao), 2) AS nota_media,
       round(100 * avg(CAST(nota_avaliacao <= 2 AS INT)), 1) AS pct_nota_baixa_1_2,
       round(100 * avg(CAST(nota_avaliacao = 5 AS INT)), 1) AS pct_nota_5
FROM base WHERE nota_avaliacao IS NOT NULL
GROUP BY 1 ORDER BY 1 DESC""")
p1

**Gradiente: a nota cai conforme o atraso aumenta?** Se a relação for causal, espera-se um
efeito *dose-resposta*: quanto maior o atraso, pior a nota.

In [ ]:
cortes = [-np.inf, -15, -8, -1, 0, 3, 7, 14, np.inf]
rotulos = ["≤ −15", "−14 a −8", "−7 a −1", "0 (no dia)", "1 a 3", "4 a 7", "8 a 14", "≥ 15"]
av = df[df.nota_avaliacao.notna()].copy()
av["faixa_atraso"] = pd.cut(av.dias_atraso, cortes, labels=rotulos)
av["nota_baixa"] = (av.nota_avaliacao <= 2).astype(int)
grad = (av.groupby("faixa_atraso", observed=True)
          .agg(pedidos=("nota_avaliacao", "size"), nota_media=("nota_avaliacao", "mean"),
               pct_nota_baixa=("nota_baixa", lambda s: 100 * s.mean())).round(2))
grad

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
cores = [G.AZUL if i < 4 else G.LARANJA for i in range(len(grad))]
barras = ax.bar(grad.index.astype(str), grad.pct_nota_baixa, color=cores, width=0.62)
for b, v in zip(barras, grad.pct_nota_baixa):
    ax.text(b.get_x() + b.get_width() / 2, v + 1.2, f"{G.br(v, 0)}%", ha="center", fontsize=8.5, color=G.TEXTO_2)
G.pct_axis(ax)
ax.set_ylim(0, max(grad.pct_nota_baixa) * 1.15)
ax.set_xlabel("Dias de atraso em relação à data prometida (negativo = entregue antes)")
ax.set_ylabel("Pedidos com nota 1 ou 2")
ax.set_title("P1 · Quanto maior o atraso, maior a chance de nota baixa")
G.subtitulo(ax, "Azul: entregue no prazo ou antes  ·  Laranja: entregue com atraso")
G.fonte(fig)
G.salvar(fig, "p1_nota_baixa_por_atraso", DESTINOS)

**Testes estatísticos.**
- **Mann-Whitney:** compara a distribuição das notas (variável ordinal) entre pedidos no prazo e atrasados.
- **Qui-quadrado:** testa a associação entre atraso e nota baixa.
- **Regressão logística:** estima o efeito do atraso **controlando** outros fatores que também
  podem irritar o cliente, como pedido com vários itens, vários vendedores, valor, peso do frete e região.

In [ ]:
no_prazo, atrasado = av[av.atraso == 0].nota_avaliacao, av[av.atraso == 1].nota_avaliacao
mw = stats.mannwhitneyu(atrasado, no_prazo, alternative="less")
tab = pd.crosstab(av.atraso, av.nota_baixa)
chi2, p_chi, _, _ = stats.chi2_contingency(tab)
risco_a, risco_n = av[av.atraso == 1].nota_baixa.mean(), av[av.atraso == 0].nota_baixa.mean()
print(f"Mann-Whitney (notas dos atrasados < notas dos no prazo): U = {mw.statistic:,.0f}; p = {mw.pvalue:.1e}")
print(f"Qui-quadrado (atraso × nota baixa): χ² = {chi2:,.0f}; p = {p_chi:.1e}")
print(f"Risco relativo de nota baixa: {risco_a:.1%} / {risco_n:.1%} = {risco_a / risco_n:.1f}×")

av["multi_itens"] = (av.qtd_itens > 1).astype(int)
av["multi_vendedores"] = (av.qtd_vendedores > 1).astype(int)
av["log_valor"] = np.log(av.valor_produtos)
av["pct_frete"] = 100 * av.valor_frete / (av.valor_produtos + av.valor_frete)
av["regiao"] = pd.Categorical(av.regiao_cliente, categories=["Sudeste", "Sul", "Centro-Oeste", "Nordeste", "Norte"])  # referência: Sudeste
logit = smf.logit("nota_baixa ~ atraso + multi_itens + multi_vendedores + log_valor + pct_frete + regiao",
                  data=av).fit(disp=False)
ors = pd.DataFrame({"odds_ratio": np.exp(logit.params), "ic95_inf": np.exp(logit.conf_int()[0]),
                    "ic95_sup": np.exp(logit.conf_int()[1]), "p_valor": logit.pvalues}).round(3)
ors.index = ors.index.str.replace(r"regiao\[T\.", "região do cliente (vs Sudeste): ", regex=True).str.replace("]", "")
ors.drop("Intercept")

In [ ]:
INDICADORES += [("P1", "nota média — no prazo", float(no_prazo.mean())), ("P1", "nota média — atrasado", float(atrasado.mean())),
                ("P1", "% nota baixa — no prazo", 100 * risco_n), ("P1", "% nota baixa — atrasado", 100 * risco_a),
                ("P1", "risco relativo nota baixa", risco_a / risco_n), ("P1", "odds ratio atraso (ajustado)", float(np.exp(logit.params["atraso"])))]

**Discussão P1:** **hipótese confirmada.**
- **Tamanho do efeito.** Pedidos atrasados têm nota média de **2,27**, contra **4,29** nos entregues
  no prazo. A proporção de notas baixas (1 ou 2) sobe de **9,2% para 62,4%**: o risco relativo é de
  **6,8 vezes**. Os dois testes rejeitam com folga a hipótese de igualdade (p < 0,001).
- **Dose-resposta.** Até a data prometida, a nota baixa fica entre 9% e 12%. Com 1 a 3 dias de atraso,
  salta para 32%; com 4 a 7 dias, para 68%; acima de uma semana, para cerca de 80%. A forma da curva
  é o padrão esperado de uma relação causal, embora uma base observacional não permita provar causalidade.
- **Assimetria.** Entregar *muito* antes do prometido quase não melhora a nota: a taxa de nota baixa
  fica em torno de 9% tanto com 8 quanto com mais de 15 dias de antecedência. **O cliente pune o
  atraso, mas não premia a antecipação.** Isso abre espaço para promessas menos folgadas (ver P5).
- **Efeito controlado.** Na regressão logística, o atraso multiplica a chance de nota baixa por
  **18,3** (IC 95%: 17,3–19,4), mesmo controlando valor, frete, número de itens e de vendedores e região.
  É de longe o maior efeito do modelo. Pedidos com vários vendedores (razão de chances de 3,5) e com
  vários itens (2,7) vêm em seguida, provavelmente porque geram entregas parciais.
- **A região perde efeito.** Depois de controlar o atraso, a região do cliente deixa de ser relevante.
  A insatisfação regional, portanto, *passa pelo* atraso. Isso justifica concentrar as perguntas
  seguintes em onde, quando e por que o atraso acontece.

---
## P2 · Onde o atraso se concentra?
*Hipótese:* rotas longas e destinos no Norte e no Nordeste atrasam mais.

In [ ]:
p2_reg = sql("""
SELECT regiao_cliente, count(*) AS pedidos,
       round(100 * avg(CAST(flag_atraso AS INT)), 2) AS taxa_atraso,
       percentile(dias_corridos_entrega, 0.5) AS prazo_real_mediano,
       percentile(dias_prometidos, 0.5) AS prazo_prometido_mediano,
       round(percentile(distancia_km, 0.5)) AS distancia_mediana_km
FROM base GROUP BY 1 ORDER BY taxa_atraso DESC""")
p2_reg

In [ ]:
p2_uf = sql("""
SELECT uf_cliente AS uf, regiao_cliente AS regiao, count(*) AS pedidos,
       round(100 * avg(CAST(flag_atraso AS INT)), 2) AS taxa_atraso
FROM base GROUP BY 1, 2 ORDER BY taxa_atraso DESC""")
p2_uf["atrasados"] = (p2_uf.pedidos * p2_uf.taxa_atraso / 100).round().astype(int)
p2_uf["pct_dos_atrasos"] = (100 * p2_uf.atrasados / p2_uf.atrasados.sum()).round(1)
print("UFs que mais concentram pedidos atrasados (volume absoluto):")
print(p2_uf.sort_values("atrasados", ascending=False).head(6).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6.4))
d = p2_uf.sort_values("taxa_atraso")
cor = [G.LARANJA if r in ("Nordeste", "Norte") else G.AZUL for r in d.regiao]
ax.barh(d.uf, d.taxa_atraso, color=cor, height=0.66)
for y, (v, n) in enumerate(zip(d.taxa_atraso, d.pedidos)):
    ax.text(v + 0.3, y, f"{G.br(v, 1)}%  (n={G.br(n, 0)})", va="center", fontsize=7.5, color=G.TEXTO_2)
ax.axvline(TAXA, color=G.TEXTO_2, lw=1, ls="--")
ax.set_ylim(-1.6, len(d) - 0.4)
ax.text(TAXA + 0.25, -1.1, f"média Brasil {G.br(TAXA, 1)}%", fontsize=8, color=G.TEXTO_2, va="center")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
G.pct_axis(ax, "x"); ax.set_xlim(0, d.taxa_atraso.max() * 1.35)
ax.set_xlabel("Taxa de atraso")
ax.set_title("P2 · Taxa de atraso por UF de destino")
G.subtitulo(ax, "Laranja: Norte e Nordeste  ·  Azul: demais regiões")
G.fonte(fig)
G.salvar(fig, "p2_atraso_por_uf", DESTINOS)

In [ ]:
cortes_km = [0, 100, 500, 1000, 2000, np.inf]
rot_km = ["< 100", "100–500", "500–1.000", "1.000–2.000", "≥ 2.000"]
dk = df[df.distancia_km.notna()].copy()
dk["faixa_km"] = pd.cut(dk.distancia_km, cortes_km, labels=rot_km, right=False)
p2_km = (dk.groupby("faixa_km", observed=True)
           .agg(pedidos=("atraso", "size"), taxa_atraso=("atraso", lambda s: 100 * s.mean()),
                prazo_real_mediano=("dias_corridos_entrega", "median"),
                prazo_prometido_mediano=("dias_prometidos", "median")).round(2))
chi2_km, p_km, _, _ = stats.chi2_contingency(pd.crosstab(dk.faixa_km, dk.atraso))
rho_km = stats.spearmanr(dk.distancia_km, dk.atraso)
print(f"Qui-quadrado (faixa de distância × atraso): χ² = {chi2_km:,.0f}; p = {p_km:.1e}")
print(f"Spearman (distância × atraso): ρ = {rho_km.statistic:.3f}; p = {rho_km.pvalue:.1e}")
p2_km

**Distância ou região?** As duas variáveis se confundem, porque o Nordeste fica longe dos vendedores,
concentrados no Sudeste. Para separar os efeitos, a taxa de atraso por região é comparada
**dentro de uma mesma faixa de distância**.

In [ ]:
estrat = (dk.pivot_table(index="faixa_km", columns="regiao_cliente", values="atraso",
                         aggfunc=lambda s: round(100 * s.mean(), 1) if len(s) >= 100 else np.nan, observed=True)
            [["Sudeste", "Sul", "Centro-Oeste", "Nordeste", "Norte"]])
estrat.columns.name = "taxa de atraso (%) por região do cliente — células com ≥ 100 pedidos"
estrat

In [ ]:
p2_rota = sql("""
SELECT regiao_vendedor, regiao_cliente, count(*) AS pedidos,
       round(100 * avg(CAST(flag_atraso AS INT)), 1) AS taxa_atraso
FROM base WHERE regiao_vendedor <> 'desconhecido'
GROUP BY 1, 2""")
ordem = ["Sudeste", "Sul", "Centro-Oeste", "Nordeste", "Norte"]
mat = p2_rota.pivot(index="regiao_vendedor", columns="regiao_cliente", values="taxa_atraso").reindex(index=ordem, columns=ordem)
n_mat = p2_rota.pivot(index="regiao_vendedor", columns="regiao_cliente", values="pedidos").reindex(index=ordem, columns=ordem)
print("Participação da origem (região do vendedor) no total de pedidos:")
print((100 * n_mat.sum(axis=1) / n_mat.values[~np.isnan(n_mat.values)].sum()).round(1).to_string())

fig, ax = plt.subplots(figsize=(7.2, 4.6))
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("azul", G.AZUL_RAMPA)
m = np.ma.masked_where(n_mat.values < 30, mat.values)
im = ax.imshow(m, cmap=cmap, vmin=0, vmax=np.nanmax(m))
for i in range(len(ordem)):
    for j in range(len(ordem)):
        n = n_mat.values[i, j]
        if not np.isnan(n) and n >= 30:
            v = mat.values[i, j]
            ax.text(j, i, f"{G.br(v, 1)}%\nn={G.br(n, 0)}", ha="center", va="center", fontsize=8,
                    color="white" if v > np.nanmax(m) * 0.55 else G.TEXTO)
        else:
            ax.text(j, i, "n < 30", ha="center", va="center", fontsize=7, color=G.TEXTO_2)
ax.set_xticks(range(5), ordem); ax.set_yticks(range(5), ordem)
ax.set_xlabel("Região do cliente (destino)"); ax.set_ylabel("Região do vendedor (origem)")
ax.grid(False); [s.set_visible(False) for s in ax.spines.values()]
ax.set_title("P2 · Taxa de atraso por rota (origem → destino)")
cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.outline.set_visible(False)
cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
G.fonte(fig)
G.salvar(fig, "p2_atraso_por_rota", DESTINOS)

In [ ]:
INDICADORES += [("P2", f"taxa de atraso — {r.regiao_cliente}", r.taxa_atraso) for r in p2_reg.itertuples()]
INDICADORES += [("P2", f"taxa de atraso — {k} km", float(v)) for k, v in p2_km.taxa_atraso.items()]

**Discussão P2:** **hipótese parcialmente confirmada: a distância importa, mas o destino importa além dela.**
- **Por região.** O Nordeste tem **12,8%** de atraso, o dobro do Sudeste (6,1%). O Norte tem 8,6%.
- **Por distância.** A taxa cresce de forma consistente com a distância: **4,5%** abaixo de 100 km e
  **12,1%** acima de 2.000 km (qui-quadrado, p < 0,001). Mas a correlação individual é fraca
  (ρ = 0,07), ou seja, a distância sozinha explica pouco de cada pedido.
- **Distância ou destino?** Comparando pedidos *na mesma faixa de distância*, o Nordeste continua pior.
  Entre 1.000 e 2.000 km, por exemplo, o atraso é de 12,9% no Nordeste contra 5,8% no Sudeste.
  Isso aponta para a **última milha** de certos destinos, e não apenas para a extensão da rota.
- **Achado inesperado: o Rio de Janeiro.** Embora esteja perto do polo vendedor (83,5% dos pedidos saem
  do Sudeste), o RJ tem **12,2%** de atraso, no nível da Bahia. Pelo volume, o RJ responde sozinho por
  **22,9% de todos os atrasos**, quase o mesmo que São Paulo (27,8%), que recebe mais que o triplo de
  pedidos. Junto com o Nordeste, o RJ é o alvo prioritário de gestão.
- **Exceção ao Norte.** Estados como AM, AP e RO têm atraso baixo (cerca de 3%), apesar das grandes
  distâncias. A P5 mostra que isso se explica por promessas mais folgadas nesses destinos.

---
## P3 · Quando o atraso acontece? Está ligado a picos de demanda?
*Hipótese:* meses de alto volume saturam a logística e elevam o atraso.

In [ ]:
p3 = sql("""
SELECT ano_mes, count(*) AS pedidos,
       round(100 * avg(CAST(flag_atraso AS INT)), 2) AS taxa_atraso,
       percentile(dias_prometidos, 0.5) AS prazo_prometido_mediano,
       percentile(dias_corridos_entrega, 0.5) AS prazo_real_mediano,
       round(avg(dias_transporte), 1) AS dias_transporte_medio,
       round(avg(dias_ate_postagem), 1) AS dias_ate_postagem_medio
FROM base GROUP BY 1 ORDER BY 1""")
rho_m = stats.spearmanr(p3.pedidos, p3.taxa_atraso)
print(f"Spearman (volume mensal × taxa de atraso, n = {len(p3)} meses): ρ = {rho_m.statistic:.2f}; p = {rho_m.pvalue:.3f}")
p3

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True, gridspec_kw={"height_ratios": [1, 1.25], "hspace": 0.35})
x = np.arange(len(p3))
destaque = p3.ano_mes.isin(["2017-11", "2018-02", "2018-03"])
a1.bar(x, p3.pedidos, color=np.where(destaque, G.LARANJA, G.CINZA), width=0.7)
a1.set_title("P3 · Volume mensal de pedidos (acima) e taxa de atraso (abaixo)")
a1.set_ylabel("Pedidos"); a1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f} mil" if v else "0"))
a2.plot(x, p3.taxa_atraso, color=G.AZUL, marker="o", ms=4)
for i in np.where(destaque)[0]:
    a2.annotate(f"{G.br(p3.taxa_atraso[i], 1)}%", (x[i], p3.taxa_atraso[i]), textcoords="offset points",
                xytext=(0, 7), ha="center", fontsize=8.5, color=G.TEXTO)
a2.axhline(TAXA, color=G.TEXTO_2, lw=1, ls="--")
a2.text(len(p3) - 0.5, TAXA + 0.6, f"média {G.br(TAXA, 1)}%", fontsize=8, color=G.TEXTO_2, ha="right")
G.pct_axis(a2); a2.set_ylabel("Taxa de atraso"); a2.set_ylim(0, p3.taxa_atraso.max() * 1.2)
a2.set_xticks(x, p3.ano_mes, rotation=60, ha="right")
G.subtitulo(a1, "Laranja: meses com os maiores picos de atraso (nov/2017, fev/2018 e mar/2018)")
G.fonte(fig)
G.salvar(fig, "p3_volume_e_atraso_mensal", DESTINOS)

**Black Friday 2017 (24/11/2017):** a comparação abaixo coloca a semana da Black Friday frente ao
restante de novembro.

In [ ]:
p3_bf = sql("""
SELECT CASE WHEN data_compra BETWEEN '2017-11-20' AND '2017-11-26' THEN 'semana da Black Friday (20–26/11)'
            ELSE 'demais dias de novembro' END AS periodo,
       count(*) AS pedidos, round(count(*) / count(DISTINCT data_compra)) AS pedidos_por_dia,
       round(100 * avg(CAST(flag_atraso AS INT)), 1) AS taxa_atraso,
       round(avg(dias_ate_postagem), 1) AS dias_ate_postagem_medio,
       round(avg(dias_transporte), 1) AS dias_transporte_medio
FROM base WHERE ano_mes = '2017-11' GROUP BY 1""")
p3_bf

In [ ]:
INDICADORES += [("P3", "spearman volume × atraso (ρ)", float(rho_m.statistic))]
INDICADORES += [("P3", f"taxa de atraso — {r.ano_mes}", r.taxa_atraso) for r in p3.itertuples()]

**Discussão P3:** **hipótese parcialmente confirmada: o volume explica um dos picos, mas não os outros.**
- **Associação.** Volume mensal e taxa de atraso têm correlação moderada (Spearman ρ = 0,57; p = 0,009;
  n = 20 meses).
- **Novembro de 2017 confirma a hipótese.** Na semana da Black Friday, a média foi de **416 pedidos por
  dia, contra 190** no restante do mês. A taxa de atraso dos pedidos dessa semana chegou a **17,3%,
  contra 9,2%**. Os dois elos ficaram mais lentos, postagem e transporte, o que caracteriza saturação
  de capacidade.
- **Fevereiro e março de 2018 contrariam a hipótese.** Esses meses tiveram as maiores taxas da série
  (**14,1% e 19,0%**) com volume parecido ao de janeiro/2018 (5,7%). O tempo de postagem ficou estável
  (cerca de 3,5 dias), mas o tempo de **transporte** subiu de 10,5 para cerca de 13 dias. Foi, portanto,
  um choque *externo* ao vendedor. Em março, o pico coincide com a **greve nacional dos Correios
  iniciada em 12/03/2018** (Agência Brasil, 2018). O pico de fevereiro não tem explicação nesta base.
- **Leitura gerencial.** Há dois tipos de risco: o **previsível**, como as datas comerciais, que se
  enfrenta com planejamento de capacidade, e o **choque externo**, como greves, que pede monitoramento
  e comunicação proativa com o cliente. Vale notar também que, a partir de abril/2018, o transporte
  ficou mais rápido e a Olist encurtou as promessas (mediana de 14 dias em agosto). Com menos folga,
  o atraso voltou a subir em agosto (6,2%).

---
## P4 · Quem causa o atraso: o vendedor ou o transporte?
*Hipótese:* o atraso nasce no vendedor, que posta o pacote depois do prazo-limite.

O ciclo do pedido tem dois elos: **compra → postagem**, que é responsabilidade do vendedor, e
**postagem → entrega**, que é responsabilidade da transportadora. A regra G04 marca como
"postagem atrasada" o pedido entregue à transportadora depois do prazo-limite dado ao vendedor.

In [ ]:
p4 = sql("""
SELECT CASE WHEN flag_postagem_atrasada THEN 'Vendedor postou atrasado' ELSE 'Vendedor postou no prazo' END AS postagem,
       count(*) AS pedidos,
       sum(CAST(flag_atraso AS INT)) AS pedidos_atrasados,
       round(100 * avg(CAST(flag_atraso AS INT)), 1) AS taxa_atraso
FROM base WHERE flag_postagem_atrasada IS NOT NULL GROUP BY 1 ORDER BY 1 DESC""")
p4["pct_dos_atrasos"] = (100 * p4.pedidos_atrasados / p4.pedidos_atrasados.sum()).round(1)
p4

In [ ]:
tempos = sql("""
SELECT CASE WHEN flag_atraso THEN 'Atrasado' ELSE 'No prazo' END AS entrega,
       count(*) AS pedidos,
       round(avg(dias_ate_postagem), 2) AS media_dias_ate_postagem,
       round(avg(dias_transporte), 2)  AS media_dias_transporte,
       round(avg(dias_entrega), 2)     AS media_dias_entrega,
       round(percentile(dias_ate_postagem, 0.5), 2) AS mediana_dias_ate_postagem,
       round(percentile(dias_transporte, 0.5), 2)  AS mediana_dias_transporte
FROM base WHERE dias_ate_postagem IS NOT NULL AND dias_transporte IS NOT NULL
GROUP BY 1 ORDER BY 1 DESC""").set_index("entrega")
exc_post = tempos.loc["Atrasado", "media_dias_ate_postagem"] - tempos.loc["No prazo", "media_dias_ate_postagem"]
exc_transp = tempos.loc["Atrasado", "media_dias_transporte"] - tempos.loc["No prazo", "media_dias_transporte"]
part_transp = 100 * exc_transp / (exc_post + exc_transp)
print(f"Dias a mais nos pedidos atrasados: postagem +{exc_post:.1f} d | transporte +{exc_transp:.1f} d "
      f"-> o transporte responde por {part_transp:.0f}% do tempo excedente")
tempos

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 2.8))
grupos = ["No prazo", "Atrasado"]
post = [tempos.loc[g, "media_dias_ate_postagem"] for g in grupos]
transp = [tempos.loc[g, "media_dias_transporte"] for g in grupos]
ax.barh(grupos, post, color=G.AZUL, height=0.5, label="Compra → postagem (vendedor)")
ax.barh(grupos, transp, left=[p + 0.15 for p in post], color=G.LARANJA, height=0.5, label="Postagem → entrega (transporte)")
for y, (p, t) in enumerate(zip(post, transp)):
    ax.text(p / 2, y, f"{G.br(p, 1)} d", ha="center", va="center", color="white", fontsize=8.5)
    ax.text(p + 0.15 + t / 2, y, f"{G.br(t, 1)} d", ha="center", va="center", color="white", fontsize=8.5)
    ax.text(p + t + 0.6, y, f"total {G.br(p + t, 1)} d", va="center", fontsize=8.5, color=G.TEXTO_2)
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_xlim(0, (post[1] + transp[1]) * 1.18); ax.set_xlabel("Dias (média)")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, borderaxespad=0.2)
ax.set_title("P4 · Onde o tempo é perdido: vendedor ou transporte?", pad=30)
G.fonte(fig)
G.salvar(fig, "p4_decomposicao_do_prazo", DESTINOS)

In [ ]:
INDICADORES += [("P4", f"taxa de atraso — {r.postagem}", r.taxa_atraso) for r in p4.itertuples()]
INDICADORES += [("P4", f"% dos atrasos — {r.postagem}", r.pct_dos_atrasos) for r in p4.itertuples()]
INDICADORES += [("P4", "% do tempo excedente no transporte", float(part_transp))]

**Discussão P4:** **hipótese refutada: o principal elo é o transporte, não o vendedor.**
- **Origem dos atrasos.** Em **72,3%** dos pedidos atrasados, o vendedor postou o pacote **dentro do
  prazo**. O atraso, portanto, nasceu depois da postagem.
- **Onde o tempo se perde.** Um pedido atrasado leva, em média, 33,9 dias, contra 11,0 de um pedido no
  prazo. Desses ~23 dias a mais, **19,9 estão no transporte** e só 3,0 na postagem. O transporte
  responde por **87% do tempo excedente**.
- **O vendedor ainda importa.** Quando o vendedor posta atrasado, a chance de o pedido atrasar sobe
  de **5,4% para 21,1%**, quase 4 vezes mais. Esses pedidos são só 8,9% do total, mas respondem por
  27,7% dos atrasos. São um alvo pequeno e de alto retorno: cobrar o prazo de postagem desses vendedores.
- **Implicação.** A maior alavanca está na **gestão das transportadoras e rotas**: acordos de nível de
  serviço por destino, troca de transportadora no RJ e no Nordeste e monitoramento do tempo de trânsito.
  A disciplina dos vendedores é uma alavanca secundária, mas barata.

---
## P5 · O prazo prometido está calibrado?
*Hipótese:* a promessa é folgada demais na média e, mesmo assim, falha em parte dos destinos.
Ou seja, está mal distribuída.

In [ ]:
folga = -df.dias_atraso   # positivo = chegou antes da data prometida
p5_resumo = pd.Series({
    "prazo prometido mediano (dias)": df.dias_prometidos.median(),
    "prazo real mediano (dias corridos)": df.dias_corridos_entrega.median(),
    "folga mediana (dias)": folga.median(),
    "% entregues com ≥ 7 dias de antecedência": 100 * (folga >= 7).mean(),
    "% entregues com ≥ 14 dias de antecedência": 100 * (folga >= 14).mean(),
    "% entregues com atraso": 100 * (folga < 0).mean()}).round(1)
p5_resumo.to_frame("valor")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
v = df.dias_atraso.clip(-50, 30)
bins = np.arange(-50.5, 31.5, 1)
ax.hist(v[v <= 0], bins=bins, color=G.AZUL, rwidth=0.85, label="No prazo ou antes")
ax.hist(v[v > 0], bins=bins, color=G.LARANJA, rwidth=0.85, label="Com atraso")
ax.axvline(-folga.median(), color=G.TEXTO_2, ls="--", lw=1)
ax.text(-49, ax.get_ylim()[1] * 0.92, f"linha tracejada = mediana\n({G.br(folga.median(), 0)} dias antes do prometido)",
        ha="left", va="top", fontsize=8.5, color=G.TEXTO_2)
ax.set_xlabel("Dias de atraso (negativo = chegou antes da data prometida; extremos agrupados em −50 e +30)")
ax.set_ylabel("Pedidos"); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f} mil" if v else "0"))
ax.legend(loc="upper right")
ax.set_title("P5 · A promessa de prazo é, em geral, muito folgada")
G.fonte(fig)
G.salvar(fig, "p5_distribuicao_folga", DESTINOS)

**Simulação de uma promessa alternativa (validação fora da amostra).** Para cada UF de destino,
a promessa passa a ser o **percentil q** do prazo real observado em 2017 (dados de "treino").
Essa regra é aplicada aos pedidos de **jan–ago/2018** ("teste") e comparada com o que a Olist
efetivamente prometeu a esses mesmos pedidos.

In [ ]:
treino, teste = df[df.data_compra < pd.Timestamp("2018-01-01").date()], df[df.data_compra >= pd.Timestamp("2018-01-01").date()].copy()
linhas = [{"regra": "Olist (real)", "prazo_prometido_medio": teste.dias_prometidos.mean(),
           "taxa_atraso": 100 * (teste.dias_corridos_entrega > teste.dias_prometidos).mean()}]
for q in [0.80, 0.85, 0.90, 0.92, 0.94, 0.95, 0.97]:
    regra = np.ceil(treino.groupby("uf_cliente").dias_corridos_entrega.quantile(q))
    prom = teste.uf_cliente.map(regra).fillna(np.ceil(treino.dias_corridos_entrega.quantile(q)))
    linhas.append({"regra": f"percentil {int(q*100)} por UF", "prazo_prometido_medio": prom.mean(),
                   "taxa_atraso": 100 * (teste.dias_corridos_entrega > prom).mean()})
p5_sim = pd.DataFrame(linhas).round(2)
p5_sim["dias_a_menos_que_olist"] = (p5_sim.prazo_prometido_medio[0] - p5_sim.prazo_prometido_medio).round(1)
p5_sim

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
r = p5_sim.iloc[1:]
ax.plot(r.prazo_prometido_medio, r.taxa_atraso, color=G.AZUL, marker="o", label="Regra: percentil do prazo real por UF")
for _, l in r.iterrows():
    ax.annotate(l.regra.split()[1], (l.prazo_prometido_medio, l.taxa_atraso), textcoords="offset points",
                xytext=(6, 4), fontsize=8, color=G.TEXTO_2)
o = p5_sim.iloc[0]
ax.scatter(o.prazo_prometido_medio, o.taxa_atraso, s=70, color=G.LARANJA, zorder=3, label="Promessa real da Olist",
           edgecolor="white", linewidth=1.5)
ax.annotate(f"Olist: {G.br(o.prazo_prometido_medio, 1)} d · {G.br(o.taxa_atraso, 1)}%", (o.prazo_prometido_medio, o.taxa_atraso),
            textcoords="offset points", xytext=(-10, -14), ha="right", fontsize=8.5, color=G.TEXTO)
G.pct_axis(ax); ax.set_xlabel("Prazo prometido médio (dias)"); ax.set_ylabel("Taxa de atraso resultante")
ax.grid(axis="x", visible=True); ax.legend(loc="upper right")
ax.set_title("P5 · Prometer menos dias custa mais atrasos")
G.subtitulo(ax, "Regra calibrada em 2017 e aplicada aos pedidos de jan–ago/2018")
G.fonte(fig)
G.salvar(fig, "p5_fronteira_prazo_atraso", DESTINOS)

In [ ]:
p5_uf = (df.groupby("uf_cliente").agg(pedidos=("atraso", "size"), taxa_atraso=("atraso", lambda s: 100 * s.mean()),
                                      prometido_mediano=("dias_prometidos", "median"),
                                      real_mediano=("dias_corridos_entrega", "median"),
                                      real_p90=("dias_corridos_entrega", lambda s: s.quantile(0.9)))
           .assign(folga_mediana=lambda d: d.prometido_mediano - d.real_mediano).round(1)
           .sort_values("taxa_atraso", ascending=False))
rho_folga = stats.spearmanr(p5_uf.folga_mediana, p5_uf.taxa_atraso)
print(f"Spearman (folga mediana da UF × taxa de atraso da UF, n = 27): ρ = {rho_folga.statistic:.2f}; p = {rho_folga.pvalue:.3f}")
p5_uf

In [ ]:
INDICADORES += [("P5", k, float(v)) for k, v in p5_resumo.items()]
INDICADORES += [("P5", f"simulação {r.regra} — prazo médio", r.prazo_prometido_medio) for r in p5_sim.itertuples()]
INDICADORES += [("P5", f"simulação {r.regra} — taxa de atraso", r.taxa_atraso) for r in p5_sim.itertuples()]

**Discussão P5:** **hipótese parcialmente confirmada: a folga é grande e mal distribuída, mas não é desperdício.**
- **A folga é grande.** A promessa mediana é de **24 dias** para uma entrega real mediana de **10
  dias**. **78,9%** dos pedidos chegam com uma semana ou mais de antecedência, e 43,4% com duas
  semanas ou mais. Como a P1 mostrou, essa antecipação não melhora a nota.
- **A folga está mal distribuída.** As UFs com menos folga são as que mais atrasam (Spearman ρ = −0,56;
  p = 0,003). Alagoas tem folga mediana de 10 dias e **21,5%** de atraso; Amazonas tem 20 dias de
  folga e **2,8%** de atraso. A promessa não acompanha a *variabilidade* de cada destino.
- **Mas a folga não é desperdício (resultado contra a hipótese).** Na simulação fora da amostra, uma
  regra por UF que prometesse o mesmo prazo médio da Olist (cerca de 23 dias) chegaria a uma taxa de
  atraso de cerca de 7,6%, praticamente igual aos 7,75% da Olist. Cortar 2,5 dias (percentil 90) faria
  o atraso subir para 10,3%. Ou seja, a Olist já opera perto da fronteira entre prazo e atraso.
  **A folga média alta é o preço da cauda longa do transporte.**
- **Implicação.** Só é possível prometer menos sem atrasar mais se a *variabilidade* do transporte cair
  (P4) ou se a promessa usar mais informação do que a UF, como rota, época do ano e risco previsto (P6).

---
## P6 · Dá para prever o risco de atraso no momento da compra? *(exploratória)*
**Desenho do experimento:**
- **Variáveis:** só as conhecidas no ato da compra: distância, mesma UF, regiões de origem e
  destino, mês e dia da semana da compra, prazo prometido, valor, frete, quantidade de itens e de
  vendedores, peso do produto.
- **Validação temporal:** o modelo é treinado com dados de 2017 e testado em jan–ago/2018, o que
  simula o uso real (prever o futuro com o passado).
- **Modelos comparados:** regressão logística (interpretável) e *gradient boosting* (não linear).
- **Métricas:** AUC-ROC, precisão média (AP) frente à taxa-base e *lift* do decil de maior risco.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

NUM = ["distancia_km", "dias_prometidos", "valor_produtos", "valor_frete", "qtd_itens", "qtd_vendedores", "peso_g"]
CAT = ["regiao_cliente", "regiao_vendedor", "mes", "dia_semana", "mesma_uf"]
X = df[NUM + CAT].copy(); X[CAT] = X[CAT].astype(str)
y = df.atraso
eh_teste = df.data_compra >= pd.Timestamp("2018-01-01").date()
Xtr, Xte, ytr, yte = X[~eh_teste], X[eh_teste], y[~eh_teste], y[eh_teste]

prep = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
modelos = {"Regressão logística": LogisticRegression(max_iter=2000, class_weight="balanced"),
           "Gradient boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=42)}
res, scores = [], {}
for nome, m in modelos.items():
    pipe = make_pipeline(prep, m).fit(Xtr, ytr)
    s = pipe.predict_proba(Xte)[:, 1]; scores[nome] = s
    top = s >= np.quantile(s, 0.9)
    res.append({"modelo": nome, "AUC_ROC": roc_auc_score(yte, s), "precisao_media_AP": average_precision_score(yte, s),
                "taxa_base_teste_%": 100 * yte.mean(), "taxa_atraso_top10%_risco": 100 * yte[top].mean(),
                "lift_top10%": yte[top].mean() / yte.mean()})
    if nome == "Regressão logística":
        nomes = pipe[0].get_feature_names_out()
        coefs = pd.Series(pipe[-1].coef_[0], index=nomes).sort_values(key=abs, ascending=False)
p6 = pd.DataFrame(res).round(3)
print(f"Treino: {len(ytr):,} pedidos (2017) | Teste: {len(yte):,} pedidos (jan–ago/2018)")
p6

In [ ]:
print("Regressão logística: 10 maiores coeficientes (variáveis padronizadas; positivo = mais risco)")
coefs.head(10).round(3).to_frame("coeficiente")

In [ ]:
melhor = p6.sort_values("AUC_ROC", ascending=False).modelo.iloc[0]
dec = pd.DataFrame({"s": scores[melhor], "y": yte.values})
dec["decil"] = pd.qcut(dec.s.rank(method="first"), 10, labels=range(1, 11))
dec = dec.groupby("decil", observed=True).y.mean().mul(100)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(dec.index.astype(str), dec.values, color=[G.LARANJA if i >= 9 else G.AZUL for i in range(10)], width=0.62)
ax.axhline(100 * yte.mean(), color=G.TEXTO_2, ls="--", lw=1)
ax.text(0, 100 * yte.mean() + 0.4, f"taxa média no teste: {G.br(100*yte.mean(), 1)}%", fontsize=8, color=G.TEXTO_2)
for i, v in enumerate(dec.values):
    ax.text(i, v + 0.3, f"{G.br(v, 1)}%", ha="center", fontsize=8, color=G.TEXTO_2)
G.pct_axis(ax); ax.set_xlabel("Decil de risco previsto (1 = menor risco, 10 = maior risco)")
ax.set_ylabel("Taxa de atraso observada")
ax.set_title(f"P6 · Taxa de atraso real por decil de risco previsto ({melhor.lower()})")
G.subtitulo(ax, "Modelo treinado com 2017 e avaliado em jan–ago/2018 (fora da amostra)")
G.fonte(fig)
G.salvar(fig, "p6_lift_por_decil", DESTINOS)

In [ ]:
INDICADORES += [("P6", f"{r.modelo} — AUC", r.AUC_ROC) for r in p6.itertuples()]
INDICADORES += [("P6", f"{m} — lift top 10%", l) for m, l in zip(p6["modelo"], p6["lift_top10%"])]

**Discussão P6:** **resultado moderado: útil para triagem, insuficiente para prever pedido a pedido.**
- **Desempenho.** A regressão logística teve **AUC = 0,67**, com precisão média de 0,14 contra uma
  taxa-base de 0,077. O *gradient boosting* foi pior (AUC 0,63), um sinal de que ele se ajustou demais
  aos padrões de 2017.
- **Valor prático.** Os 10% de pedidos com maior risco previsto atrasaram **15,8%** das vezes, o
  **dobro** da média (lift de 2,0). Os 10% de menor risco atrasaram só 1,0%. Isso basta para
  **priorizar** ações, como monitorar esses pedidos, alertar o cliente ou ampliar a promessa, mas não
  para garantir prazos individualmente.
- **O que o modelo aprendeu.** Os maiores pesos estão na sazonalidade (compras em novembro e
  dezembro), na rota (mesma UF protege) e no destino Nordeste. Prazos prometidos maiores reduzem o
  risco, o que é coerente com a P5.
- **Limitação.** Os maiores picos do teste (fev–mar/2018) vieram de um choque externo que não estava
  nos dados de treino. Nenhuma variável disponível no ato da compra antecipa uma greve. Dados
  operacionais, como a transportadora usada e o status de rastreio, ficam como trabalho futuro.

---
## Persistência dos resultados
Os indicadores que respondem às perguntas também são gravados na camada Gold, como evidência
reprodutível dos resultados.

In [ ]:
ind = pd.DataFrame(INDICADORES, columns=["pergunta", "indicador", "valor"])
ind["valor"] = ind.valor.astype(float).round(4)
with C.Stage("05_analise", "gold.resultados_indicadores") as st:
    st.rows = C.write_table(spark.createDataFrame(ind), "gold", "resultados_indicadores")
ind

---
## Discussão geral
A hipótese central foi **confirmada em sua maior parte**, com nuances que mudam a recomendação:

1. **O atraso é o principal fator de insatisfação medido.** Ele multiplica por cerca de 18 a chance de
   nota baixa (P1), e entregar mais cedo do que o prometido não compensa.
2. **O atraso não é aleatório.** Ele se concentra em **destinos** (Nordeste e Rio de Janeiro, mesmo
   comparando distâncias iguais; P2), em **períodos** (picos de demanda, como a Black Friday, e choques
   externos, como a greve de março/2018; P3) e em **um elo da cadeia**, o transporte, que concentra
   87% do tempo excedente (P4).
3. **O atraso é gerenciável, mas não pelo caminho que se esperava.** O vendedor não é o principal
   culpado (P4), e a promessa de prazo já está perto do ótimo para uma regra simples (P5). A alavanca
   principal é **reduzir a variabilidade do transporte** nas rotas críticas, e a previsão de risco (P6)
   ajuda a priorizar onde agir.

**Plano de ação sugerido pelos dados:**

| Prioridade | Ação | Evidência |
|---|---|---|
| 1 | Acordo de nível de serviço com transportadoras, por destino, focado em RJ e Nordeste | P2, P4 |
| 2 | Plano de capacidade para datas comerciais (Black Friday) | P3 |
| 3 | Promessa de prazo por rota e época do ano, com folga proporcional à variabilidade | P5, P6 |
| 4 | Alerta proativo ao cliente nos pedidos do decil de maior risco | P1, P6 |
| 5 | Cobrança do prazo de postagem aos vendedores reincidentes (8,9% dos pedidos, 27,7% dos atrasos) | P4 |

**Viabilidade financeira do MVP:** o pipeline inteiro roda em infraestrutura gratuita, com o Google Colab
como processamento e o Google Drive (15 GB gratuitos) como armazenamento, para cerca de 126 MB de dados
brutos. Em produção, o mesmo desenho caberia em uma conta de nuvem de baixo custo. O volume de
~100 mil pedidos em 2 anos não exige processamento distribuído; o Spark foi usado pela escalabilidade
e pela aderência à disciplina.